# 04 — Inférence

Ce notebook charge un checkpoint et traduit des phrases libres en anglais → français via beam search.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("seq2seq-attention"):
        !git clone https://github.com/Arsenicophe/seq2seq-attention.git
    os.chdir("seq2seq-attention/notebooks")
    !pip install -q sacrebleu rouge-score spacy
    !python -m spacy download en_core_web_sm -q
    !python -m spacy download fr_core_news_sm -q

print("Répertoire courant :", os.getcwd())

## Imports

In [ ]:
import sys, yaml
sys.path.append("../src")

import torch
import spacy
from datasets import load_dataset
from tqdm import tqdm

from encoder  import Encoder
from decoder  import Decoder
from seq2seq  import Seq2Seq
from data     import Vocab, EOS_IDX
from sampling.beam_search import beam_search_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

## Chargement du checkpoint

In [ ]:
CHECKPOINT_PATH = "../checkpoints/checkpoint_epoch_10.pt"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# Reconstruction du vocab (identique à l'entraînement)
nlp_en = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

def batch_tokenize(nlp, sentences, batch_size=512):
    return [
        [tok.text.lower() for tok in doc]
        for doc in tqdm(nlp.pipe(sentences, batch_size=batch_size), total=len(sentences))
    ]

N    = cfg["data"]["n_train"]
ds   = load_dataset("Helsinki-NLP/opus-100", "en-fr")
train = ds["train"]

src_train = [ex["translation"]["en"] for ex in train.select(range(N))]
trg_train = [ex["translation"]["fr"] for ex in train.select(range(N))]

src_vocab = Vocab.build(batch_tokenize(nlp_en, src_train), min_freq=cfg["data"]["min_freq"])
trg_vocab = Vocab.build(batch_tokenize(nlp_fr, trg_train), min_freq=cfg["data"]["min_freq"])

# Reconstruction du modèle
encoder = Encoder(vocab_size=len(src_vocab), **cfg["encoder"])
decoder = Decoder(vocab_size=len(trg_vocab), **cfg["decoder"])
model   = Seq2Seq(encoder, decoder, device).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Modèle prêt.")

## Fonction de traduction

In [ ]:
nlp_en = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def translate(sentence: str, beam_size: int = 5, max_len: int = 50) -> str:
    """
    Traduit une phrase anglaise en français.

    Étapes :
        1. Tokeniser avec spaCy
        2. Convertir en indices via src_vocab
        3. Appeler beam_search_decode
        4. Convertir les indices de sortie en mots via trg_vocab
        5. Joindre en une string
    """
    # 1. Tokenisation
    tokens = [tok.text.lower() for tok in nlp_en(sentence)]

    # 2. Tokens → indices + EOS
    src_ids    = src_vocab.encode(tokens) + [EOS_IDX]
    src_tensor = torch.tensor(src_ids, device=device).unsqueeze(1)  # (src_len, 1)
    src_length = torch.tensor([len(src_ids)])

    # 3. Beam search
    model.eval()
    pred_ids = beam_search_decode(
        model, src_tensor, src_length,
        beam_size=beam_size, max_len=max_len, device=device
    )

    # 4. Indices → tokens (sans tokens spéciaux)
    pred_tokens = [trg_vocab.itos[i] for i in pred_ids if i not in (0, 1, 2, 3)]

    # 5. Reconstruction de la phrase
    return " ".join(pred_tokens)

## Test de traduction

In [ ]:
phrases = [
    "The cat is on the table.",
    "I love learning machine learning.",
    "How are you today?",
]

for phrase in phrases:
    print(f"EN : {phrase}")
    print(f"FR : {translate(phrase)}")
    print()